# Sprint 3 · Registry, drift and response

Optional local companion to the [Campus journey](https://github.com/richhiey/ml-llm-ops/blob/main/docs/course/README.md). Keep your source/terminal project and observations. Independent cells require your own attempt before collapsed feedback. Figures come from executed synthetic reference data and never replace your raw evidence.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
import numpy as np
import pandas as pd
from IPython import get_ipython
from IPython.display import display

# Register real PNG figure rendering in this notebook, including core Figure objects.
get_ipython().run_line_magic("matplotlib", "inline")
from linden_mlops import Config, Workspace, register, approve, activate, batch_predict, reproduce
from linden_mlops.project import Project
from linden_mlops.lifecycle import ROOT, MODEL_NAME, write_json
from linden_mlops.pipeline import DATA, sha256
from linden_mlops.operations import (
    monitor, train_candidate, compare_candidate, promote_candidate,
    write_incident, plot_monitoring,
)

# A notebook kernel may start in notebooks/. Verify its checkout, then use its app root.
opening_directory = Path.cwd().resolve()
if (opening_directory / "pyproject.toml").is_file():
    expected_root = opening_directory
elif opening_directory.name == "notebooks" and (opening_directory.parent / "pyproject.toml").is_file():
    expected_root = opening_directory.parent
else:
    expected_root = None
if expected_root is None or ROOT.resolve() != expected_root:
    raise RuntimeError("Choose this clone's .venv and start the notebook in its clone root or notebooks folder; the imported application must belong to that clone.")
os.chdir(ROOT)
project = Project()
workspace = Workspace(project.tracking)
print("Application source / working directory:", ROOT, Path.cwd())
print("Saved project:", project.work)

## A-S3-01 · From "which file is production?" to a registry

🏷️
# From "which file is production?" to a registry

Linden Facilities has chosen a run using comparable validation results. Its fitted model already exists. Now a planner asks which saved model tomorrow’s batch will load, and an engineer answers with two files: `model-final.pkl` and `model-final-new.pkl`.

A filename does not reliably distinguish what the artifact is, whether it passed the required checks and whether it is currently selected. Those are separate questions.

## Give each question its own record

A **model registry** is a catalog of named models and their versions. A **registered model**, here `dev.linden-demand`, groups the versions for one named purpose. Each **version** identifies a logged fitted artifact and the training run that produced it. Registering an existing artifact does not train another model.

A **tag** is descriptive metadata attached to a version, such as `validation_status=approved`. An **alias** is a name pointing to one chosen version. It is mutable, meaning it can be reassigned. Our alias is `active`.

| Question | Record to inspect | What that record cannot prove alone |
|---|---|---|
| Which artifact is this? | Version → source run → saved model | That its checks passed |
| What validation decision was recorded? | Status/evidence tags | That tests actually ran or selection moved |
| What will the next job choose? | Current `active` alias | What an earlier job loaded |

The saved model’s **URI** is the address used to load it. The run-to-artifact connection already exists from tracking; registration adds a version identity for lifecycle decisions, meaning decisions about validation, selection and later replacement.

![Registered artifacts retain their identities and approval tags while the active pointer chooses one version](https://github.com/richhiey/ml-llm-ops/blob/main/docs/course/sprint-3/01-from-which-file-is-production-to-a-registry/assets/from-which-file-is-production-to-a-registry__01__en.png)

## Approval and selection require different actions

The scaffold registers a pending version. After checking its validation result and input contract, an engineer records approval and a reason. The helper permits `active` to point only to an approved version. This is the course application’s policy; MLflow tags are metadata, rather than automatic proof that checks ran.

Each new batch resolves `active` once at its start, then uses that fixed version for the job. Moving the alias changes the next resolution. It does not retrain a model, delete the prior artifact or prove that an already-running consumer reloaded anything. The batch’s own metadata records the version and source run it actually used.

## Translate older stage vocabulary

Older tutorials may move a model through `Staging`, `Production` and `Archived`. Those stage labels are superseded vocabulary. Modern workflows use aliases for named selections, tags for validation information and environment-specific registered models when separate development/production organization is needed.

The `dev.` prefix in our name is a teaching convention. A name alone does not configure permissions or isolate infrastructure. Translate an older tutorial’s intended behavior into explicit validation and selection decisions instead of copying the stage label and assuming it deploys the model. See the [MLflow registry workflow](https://mlflow.org/docs/3.5.1/ml/model-registry/workflow/).

## Keep history as well as current state

A current alias lookup answers “what is selected now?” To explain a change, retain the previous version, new version, source run, time and reason. These connections are **lineage**: a path from a result or decision to what produced it.

The helper appends an event to `audit.jsonl`, a text file with one JSON event per line. Its limits are concrete: a direct edit through another interface can bypass the helper; the local file can be edited; and the alias can move even if writing its event then fails. Compare current registry state, retained event and new job metadata rather than treating any one as the entire history.

## Check for Understanding

A colleague registers a new version and announces that tomorrow’s batch will use it. The old version still has `active`. Diagnose the handover and specify the evidence and separate action needed before that claim is justified. Then translate an older instruction, “move the validated model from Staging to Production”, into a version, validation tags and an alias action. Explain what historical record you would retain.

<details><summary>Check after diagnosing the selection</summary>

**Result you should get:** registration has created an identity, but has not moved the alias. A new batch still resolves the old version. Inspect the new version’s source artifact and validation evidence before deciding whether it should be selected.

**How:** validate under the application policy, record approval, explicitly reassign `active`, and inspect a newly executed batch’s version and source run. Retain the change event with previous/new versions, source run, time and reason. Translate the older stage instruction into approval evidence on a named version and assignment of the intended alias. Separate environment-specific registered models when isolation is needed; a `dev.` name alone does not create that isolation. An approved tag alone is insufficient evidence that all of those actions occurred.

</details>

## Learning outcome

Explain 3 problems a model registry solves that ad hoc file-saving does not - versioning, alias/tag-based lifecycle control, and auditability - and recognise staging, production, and archived as superseded stage vocabulary replaced by aliases, tags, and environment-specific registered models.

## Summary

You can separate which artifact exists, what validation was recorded and what a new job will select. Linden can now replace a selection without losing the identity or history of the model it replaces.


## A-S3-02 · Registering and promoting a model

📝
# Registering and promoting a model

Linden Facilities has selected a guarded validation run. We will register that run’s fitted artifact, record the validation decision and select it for a new batch. No additional model training is needed.

## Starting point and finished state

Use the existing VS Code clone, saved `selection` in `work/project.json` and local tracking store. A **model version** identifies a registered artifact; its **tags** describe validation; the mutable `active` **alias** selects which version a new job loads. A model **URI** is its saved loading address.

Finish with a usable planner CSV and a traceable run → version → new batch connection, plus the deliberate rejection of an attempt to activate a pending version. The commands are ordinary application operations. Keep the actual run and version IDs from their outputs; the example labels below must be replaced with your observed values.

## Follow these steps

### Step 1 — Register the selected artifact

```sh
uv run linden-mlops status
uv run linden-mlops register --run-id ACTUAL_SELECTED_RUN_ID
uv run linden-mlops inspect
```

Check the selected run ID in `status`. `register` returns that ID and a version string. `inspect` prints registered versions and their tags; the new version has `validation_status=pending`. The version’s `run_id` must match the selection. Registration adds an identity to an already logged artifact.

Keep the returned identities. Reopening this activity reuses the saved version; local version counters are not universal identifiers. If it is already approved, inspect your retained first-run evidence rather than claiming it is newly pending.

### Step 2 — Observe the pending boundary fail

For the newly pending version, deliberately try:

```sh
uv run linden-mlops activate --version ACTUAL_REGISTERED_VERSION --reason 'Deliberate pending-status probe'
```

Expect a nonzero exit and `Only approved versions may be active`. This is an intentionally rejected action. Check the current state before continuing; a new registration must not have selected the candidate. The helper enforces this approval policy; a plain MLflow tag does not enforce it by itself.

### Step 3 — Inspect checks before recording approval

```sh
uv run pytest tests/test_batch_input.py tests/test_spine.py::test_contract_extension_and_unchanged_pipeline -q
uv run linden-mlops compare
```

Expect the saved-source contract test to pass. Your batch-input tests check the added `validate_planned_batch` stage, including a partial plan rejection, identity/completeness and valid-input preservation. The existing spine test separately checks domain rejection and unchanged training behavior. The student-created `src/linden_mlops/batch_input.py` and `tests/test_batch_input.py` must be present in the selected source, not only in an unsaved editor buffer. In the selected run’s raw record, check `range_guard=True`, `evaluation_partition=validation` and finite `rmse_kwh` at most `5` kWh. **Finite** means a real numeric value, rather than `NaN` or infinity.

The `5` kWh limit is the scaffold’s teaching approval gate. It is separate from choosing the smallest error among the five comparable candidates. Passing the gate does not mean every production concern has been tested.

Only after observing the checks, record your own reason:

```sh
uv run linden-mlops approve --version ACTUAL_REGISTERED_VERSION --reason 'REPLACE_WITH_YOUR_OBSERVED_CHECKS_AND_SCORE'
uv run linden-mlops inspect
```

Replace the placeholder with the tests and selected score you actually inspected. Expect `validation_status=approved` and `validation_evidence` containing your rationale. The measured score remains inspectable in the source run. `approve` checks the helper’s metric/guard policy and records your reason; it does not execute the test command for you.

### Step 4 — Select it, then inspect a new batch

```sh
uv run linden-mlops activate --version ACTUAL_REGISTERED_VERSION --reason 'REPLACE_WITH_YOUR_EVIDENCE_BASED_SELECTION_REASON'
uv run linden-mlops batch
uv run linden-mlops inspect
```

The activation event names previous and new versions, source run, time and reason. The batch resolves `active` once, loads that version and writes `work/tracking/batch.csv` and `batch.json`. Use a new `--output-name` for a distinct later job; reopen a retained job instead of overwriting it. Check that its `resolved_version` equals the registered version and its `run_id` equals the selected run. Also inspect `24` aligned finite predictions and `complete_within_60s=True`.

![The selected artifact is registered pending, checked and approved, explicitly selected and recorded by a new batch](https://github.com/richhiey/ml-llm-ops/blob/main/docs/course/sprint-3/02-registering-and-promoting-a-model/assets/registering-and-promoting-a-model__01__en.png)

The reference diagram shows one actual connected execution. Your IDs may differ. The next job’s metadata proves what it loaded; a later alias lookup alone cannot prove what an earlier job used. An alias change does not force an already-running process to reload.

## Watch out for

If version, run and batch disagree, compare full IDs and store paths before approving anything else. Keep existing artifacts. The helper’s `audit.jsonl` contains one JSON event per line, but only for helper-mediated changes. The file can be edited, and an alias change can succeed even if saving its event fails. Report and reconcile such a mismatch rather than claiming a complete history.

See the [source/API lookup](https://github.com/richhiey/ml-llm-ops/blob/main/docs/api-reference.md) for the registry client and each helper’s exact acceptance conditions.

## Quick reference

| Command | Intended effect | Check |
|---|---|---|
| `register` | Create/reopen candidate version | Correct source run, initially pending |
| `approve` | Record validation under the local gate | Honest reason and observed checks |
| `activate` | Move `active` and append an event | Actual old/new version and source run |
| `batch` | Load the selected version for a job | New job’s own version/run and output |

## Check your result

Explain “registered”, “approved” and “selected” by pointing to their different records. Retain the pending rejection and the run/version/new-batch identities in your existing notes. The approval reason must describe checks you actually observed. A copied claim that tests passed is not test evidence. Do not repeat the lifecycle simply to generate another screenshot.

## Learning outcome

Use the provided model registry scaffold to register a new model version, record its validation status with tags, and promote it by assigning the scaffold's active-model alias.

## Summary

Linden’s selected training result is now a registered, validated selection backed by a new batch. You can show which action created each record and which artifact the job actually used.


In [ ]:
selection = project.selected()
registry = project.data.get("registry")
if registry and registry["run_id"] == selection["run_id"]:
    version = str(registry["version"])
else:
    version = register(workspace, selection["run_id"])
    project.save(registry={"version": str(version), "run_id": selection["run_id"]})
registered = workspace.client.get_model_version(MODEL_NAME, str(version))
if registered.tags.get("validation_status") != "approved":
    try:
        activate(workspace, version, "Attempt before approval: expected rejection")
    except ValueError as error:
        print("Pending activation rejected:", error)
    else:
        raise AssertionError("A pending version became active")
    # Approval cites observations actually made here, including the saved new stage.
    assert Path("tests/test_batch_input.py").is_file(), "Complete and save the planned-batch stage tests first"
    checked = subprocess.run([sys.executable, "-m", "pytest", "tests/test_spine.py::test_contract_extension_and_unchanged_pipeline", "tests/test_batch_input.py", "-q"], cwd=ROOT, text=True, capture_output=True)
    print(checked.stdout)
    if checked.returncode:
        raise RuntimeError(checked.stderr or checked.stdout)
    run = workspace.client.get_run(selection["run_id"])
    rmse = run.data.metrics["rmse_kwh"]
    assert run.data.params["evaluation_partition"] == "validation" and np.isfinite(rmse)
    reason = f"Observed saved-source input tests passed; guarded validation RMSE {rmse:.6f} kWh satisfies the declared bound."
    approve(workspace, version, reason)
proof_name = None
for name in ["registry-check", "batch"]:
    path = workspace.path / f"{name}.json"
    if path.exists():
        retained = json.loads(path.read_text())
        if retained["run_id"] == selection["run_id"] and retained["resolved_version"] == str(version):
            proof_name, job = name, retained
            break
if proof_name is None:
    from mlflow.exceptions import MlflowException
    try:
        current = workspace.client.get_model_version_by_alias(MODEL_NAME, "active")
    except MlflowException:
        current = None
    if current is not None and current.run_id != selection["run_id"]:
        raise ValueError("An operational model is already active. Reopen the original registry job; do not reset active to replay this example.")
    if current is None or str(current.version) != str(version):
        print(activate(workspace, version, "Observed pending rejection, saved-source tests and validation acceptance before selection"))
    _, job = batch_predict(workspace, output_name="registry-check")
    proof_name = "registry-check"
predictions = pd.read_csv(workspace.path / f"{proof_name}.csv")
planned = pd.read_csv(DATA / "batch-v1.csv")
assert job["run_id"] == selection["run_id"] and job["resolved_version"] == str(version)
assert predictions.row_id.tolist() == planned.row_id.tolist() and len(predictions) == 24
assert np.isfinite(predictions.prediction_kwh).all() and job["complete_within_60s"]
assert job["input_sha256"] == sha256(DATA / "batch-v1.csv")
display(predictions)
print(job)

## A-S3-03 · Recognising drift and choosing a response

🏷
# Recognising drift and choosing a response

Linden Facilities has selected a model for its 24-row planning batch. A later input window contains warmer temperatures. The input guard still accepts every row. Does that mean the model is wrong, or that the building simply needs different predictions?

## Three signals answer different questions

**Data drift** is a change in the distribution of incoming feature values relative to a declared reference window. **Prediction drift** is a change in the output distribution; compare both windows using the same pinned model version so a model change is not the explanation. **Performance degradation** is worse measured prediction quality on labeled examples, where the actual `demand_kwh` is known. Neither of the first two establishes the third.

| Evidence | What it supports | What is still missing |
|---|---|---|
| Valid temperatures shift by `8` °C | Inputs changed | Whether demand errors increased |
| Predictions shift using one model | Its output distribution changed | Actual demand labels |
| Labeled validation RMSE exceeds the policy bound | The model misses the required quality on that cohort | Whether a candidate fixes it |
| Invalid `hour=24` | Input-contract failure | Repair the source; do not call invalid data acceptable drift |

The teaching monitor uses **Wasserstein distance**, the amount of movement between two distributions. Raw distances have the column's units: °C, hours, occupancy percentage points or kWh. A threshold is a declared action boundary, not a probability of model failure. Review the [operations source](https://github.com/richhiey/ml-llm-ops/blob/main/src/linden_mlops/operations.py) and [SciPy definition](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.wasserstein_distance.html).

![Observed drift distances relative to their declared thresholds](https://github.com/richhiey/ml-llm-ops/blob/main/docs/course/sprint-3/03-recognising-drift-and-choosing-a-response/assets/operations-drift__01__en.png)

The figure is an actual synthetic reference execution. In these supplied windows, every reference temperature was shifted by `8` °C. The one-dimensional Wasserstein distance is therefore approximately `8` °C: the distribution moved that far, even though its shape did not change. Dividing by the declared `3` °C threshold gives `8 / 3 ≈ 2.67`, the plotted ratio. This ratio has no units; it is not a `267%` probability that the model failed. A ratio of `1` reaches the configured boundary. The chart does not measure prediction error, causal change or statistical certainty.

## Decide before you execute

A **response policy** joins a signal, threshold and action. The provided policy requires at least `96` observations per monitoring window. Its distance boundaries are `3` °C, `3` hours, `8` occupancy percentage points and `5` kWh. With at least `32` separately labeled validation examples, RMSE above `3` kWh permits training a candidate. A candidate must reach RMSE at most `3` kWh and improve by at least `0.5` kWh on the same validation rows. These are bounded teaching criteria, not universal production thresholds.

Check the evidence in order: enough monitoring rows, any distance flags, enough independent labels, then measured error. **RMSE**, root mean squared error, summarizes prediction error in kWh; larger errors receive more weight. No labels means no measured RMSE, even when the distance ratio is large.

Insufficient monitoring data means collect more. Drift without sufficient labeled degradation means investigate or re-evaluate. Confirmed degradation permits a candidate; approval and activation are separate decisions. Retain the incumbent if the candidate fails. Roll back only when a version is already active and an approved alternative satisfies the recovery requirement. Avoid training on model-generated predictions as if they were true demand labels: that feedback loop can reinforce the model's errors.

## Translate the idea to an LLM system

**Model drift** can mean a changed input-to-target relationship or measured quality loss. Name which claim the evidence supports. In an LLM application, the output also depends on the prompt: the instructions supplied to the model. A **behavior baseline** is a versioned set of representative requests, expected behavior and evaluation criteria. Changes in requests, prompts and models need separate records.

Imagine Linden's support assistant extracts building names and requested planning hours. It uses prompt `p1` with model `m1`. Its held-back evaluation cases require the requested hours to be preserved and unsupported building names to be flagged.

| Observation | What changed? | What would establish degradation? | Proportionate response |
|---|---|---|---|
| More incoming requests concern unfamiliar building names; `p1` and `m1` are unchanged | Request mix; an input-distribution change | Evaluate representative new requests against known expected behavior | Investigate coverage and collect labeled examples; do not assume a prompt/model defect |
| Prompt `p2` says “use tomorrow when no date is given”; the same requests now produce different dates | A known instruction change | Compare `p1`/`p2` with the same model on fixed cases and an explicit date-handling requirement | Keep or revert the prompt according to that requirement, then verify the resulting behavior |
| `p1` is unchanged, but model `m2` drops requested hours on the same fixed cases | A model-version change with measured behavior loss | Recorded case-level failures under identical prompt, cases and criteria | Retain/recover a qualifying version or evaluate a remedy before release |

**Prompt/behavior drift** describes a change relative to that declared behavior baseline. A changed output is an observation; a failed requirement is a quality judgment. Replaying identical cases makes a prompt/model comparison interpretable, just as scoring two demand models on the same validation rows does. A prompt edit, model replacement and numeric-model retraining are different remedies. None follows automatically from an alert. This comparison needs no new LLM service or framework.

## Try a diagnosis

The same-model prediction distribution changed, but labels have not arrived. Write the signal, the missing evidence and the next action. Also explain why a valid warmer batch should not be rejected by the range guard.

For the assistant, new building-name requests arrive after a prompt deployment. You have changed outputs but no case-level evaluation. State which versions and request sets you must compare before blaming the prompt. Give one observation that would justify reverting it and one that would instead call for better request coverage.

<details>
<summary>Check after recording your own decision and evidence</summary>

**Result you should get:** Investigate the changed inputs and seek suitable labels or re-evaluation evidence; retain the current active model.

**How:** A changed output distribution cannot establish RMSE without actual targets. The guard checks domain validity, while the monitor checks distributions. Retraining immediately or clipping valid temperatures confuses these contracts.

For the assistant, keep the model and evaluation requests fixed while comparing the recorded old/new prompts. Separately evaluate representative new requests with expected outputs. Revert the new prompt if it introduces date/hour failures on the same cases under the unchanged requirement; a request-mix gap alone calls for coverage investigation. Without versions and matched cases, the two explanations remain confounded.

</details>

## Learning outcome

Distinguish data drift, prediction drift and measured performance degradation; compare model drift with prompt or behavior drift; and justify investigation, re-evaluation or retraining from the available evidence.

## Summary

Linden can distinguish a changed distribution from demonstrated model error. Its policy names the evidence needed before a candidate is trained; an alert alone does not change the active model.


## A-S3-04 · From a monitoring signal to an evaluated candidate

📝
# From a monitoring signal to an evaluated candidate

Linden Facilities receives a warmer planning window and, later, independent labeled feedback from the same simulated condition. You will operate its monitoring and response workflow, producing a decision record, a candidate comparison and a new planner batch.

## Starting point and useful result

Use the existing clone and guarded, approved active model. Keep the original five-run selection fixed. The **active alias** selects a registered version; the monitor resolves that version once for both windows. The response must remain attached to the version it observed.

The supplied [Linden intake](https://github.com/richhiey/ml-llm-ops/tree/main/docs/course/exercises/monitoring/linden) contains synthetic reference/current cohorts, labeled feedback, a policy and a data card. It is incoming task data, not real building traffic. Generating a classroom case is separate from operating the application.

Start `uv run python` from the clone root. Execute the following Python blocks in order in the same session. They call the application's reusable functions directly; the [operations source](https://github.com/richhiey/ml-llm-ops/blob/main/src/linden_mlops/operations.py) shows the policy and evidence checks behind each call. Retain actual arguments, returned IDs and observations in your operational notes. On resuming, reopen retained JSON and IDs; do not repeat completed training or overwrite a job.

## 1. Inspect the inputs and pin the incoming selection

```python
from pathlib import Path
import json
import numpy as np
import pandas as pd
from linden_mlops import Workspace, batch_predict
from linden_mlops.pipeline import DATA, sha256
from linden_mlops.lifecycle import MODEL_NAME
from linden_mlops.operations import (
    monitor, train_candidate, compare_candidate,
    promote_candidate, write_incident,
)

workspace = Workspace("work/tracking")  # Use your existing tracking location.
inputs = Path("docs/course/exercises/monitoring/linden")
outputs = Path("work/operations")
outputs.mkdir(parents=True, exist_ok=True)
print(json.loads((inputs / "data-card.json").read_text()))
print(json.loads((inputs / "policy.json").read_text()))
incumbent = workspace.client.get_model_version_by_alias(MODEL_NAME, "active")
print("Incoming version/source:", incumbent.version, incumbent.run_id)
```

Expect `192` reference and `192` current feature-only rows, with different IDs. The stable control copies the reference features; the warmer window changes temperature by `8` °C inside valid bounds. Feedback has `168` training, `48` validation and `24` reserved final rows. Its changed target relationship is an explicit simulation. Training and evaluation IDs differ. Keep final rows out of the response and do not edit thresholds after seeing scores.

Why inspect first? A distribution difference has no interpretable action until the model, population, units and policy are known.

## 2. Observe before labels arrive

```python
unlabeled = monitor(
    workspace,
    inputs / "reference.csv", inputs / "current.csv",
    inputs / "policy.json", outputs / "monitor.json",
)
print(json.dumps(unlabeled, indent=2))
```

Expect `action=investigate`, the incoming model's version/source, input hashes and distances with thresholds. There is no observed RMSE. The command does not train or change `active`. State what additional evidence would justify a candidate before continuing.

## 3. Inspect independent labeled evidence

```python
labeled = monitor(
    workspace,
    inputs / "reference.csv", inputs / "current.csv",
    inputs / "policy.json", outputs / "labeled-monitor.json",
    feedback_path=inputs / "feedback.csv",
)
print(json.dumps(labeled, indent=2))
```

Expect `48` labeled validation rows and `action=retrain_candidate` in this supplied condition. Read the actual validation RMSE and compare it with the declared `3` kWh bound. Validation labels measure error; training labels fit the candidate; final labels do neither here. The labeled rows are independent observations from the simulated condition, not targets secretly added to the feature-only monitoring window.

## 4. Train only after the trigger, then compare

```python
candidate = train_candidate(
    workspace, outputs / "labeled-monitor.json",
    inputs / "feedback.csv", alpha=1.0,
)
(outputs / "candidate.json").write_text(json.dumps(candidate, indent=2) + "\n")
print(candidate)
comparison = compare_candidate(
    workspace, outputs / "labeled-monitor.json",
    inputs / "feedback.csv", candidate["version"], outputs / "comparison.json",
)
print(json.dumps(comparison, indent=2))
roles = pd.read_csv(inputs / "feedback.csv", usecols=["row_id", "partition"])
training_ids = set(roles.loc[roles.partition.eq("train"), "row_id"])
validation_ids = set(roles.loc[roles.partition.eq("validation"), "row_id"])
assert set(comparison["evaluation_rows"]) == validation_ids
assert validation_ids.isdisjoint(training_ids)
print("Held-fixed validation rows:", len(validation_ids))
print("Error improvement in kWh:", comparison["improvement_kwh"])
```

This demonstrated configuration produces a pending candidate. Registration alone does not select it. Check that the incumbent and candidate errors use exactly the same `48` validation IDs, disjoint from candidate training. Explain both the candidate's absolute error and its improvement in kWh. The declared requirements are RMSE at most `3` kWh and improvement at least `0.5` kWh; a newer version number is irrelevant.

![Candidate and incumbent errors measured on the same labeled validation cohort](https://github.com/richhiey/ml-llm-ops/blob/main/docs/course/sprint-3/04-from-a-monitoring-signal-to-an-evaluated-candidate/assets/operations-comparison__01__en.png)

This is an executed synthetic author reference. Your report establishes your actual result and IDs. No final scores appear in the figure.

## 5. Make the selection decision and serve the planner

Use the comparison to choose the action. A passing candidate can be promoted with your observed reason. A failed candidate stays inactive; retaining the incumbent avoids an unsupported replacement, but does not prove its observed degradation disappeared. Execute the appropriate branch, then inspect a new job from that resulting version:

```python
if comparison["passed"]:
    reason = input("Observed validation and improvement supporting this decision: ").strip()
    print(promote_candidate(workspace, outputs / "comparison.json", reason))
    intended_version = str(candidate["version"])
    intended_run = candidate["run_id"]
else:
    intended_version = str(labeled["model_version"])
    intended_run = labeled["run_id"]
    print("Candidate rejected; incumbent retained. Current-condition quality remains unresolved.")
predictions, job = batch_predict(workspace, output_name="response-batch")
planned = pd.read_csv(DATA / "batch-v1.csv")
assert job["resolved_version"] == intended_version and job["run_id"] == intended_run
assert job["completed_at"] >= comparison["observed_at"]
assert predictions.row_id.tolist() == planned.row_id.tolist() and len(predictions) == 24
assert np.isfinite(predictions.prediction_kwh).all() and job["complete_within_60s"]
assert job["input_sha256"] == sha256(DATA / "batch-v1.csv")
print(job)
print(predictions)
```

Promotion recomputes acceptance from retained observations and models. An edited `passed` field cannot authorize it. A changed active version makes the trigger stale and stops the response. If acceptance fails, retain the incumbent and the failed comparison; do not force promotion. Run the new response batch from the resulting intended version.

Inspect `resolved_version` and `run_id` against the actual decision. Independently check all `24` IDs once in input order, finite predictions, matching input hash and completion within `60` seconds. The CSV is the useful planner output; the job record establishes which model served it. Use a distinct output name for a later job and retain the original evidence.

## 6. Retain an incident another engineer can use

```python
explanation = input("Why did the evidence justify this response? ").strip()
follow_up = input("Owner, next check and observable success condition: ").strip()
incident = write_incident(
    workspace, outputs / "labeled-monitor.json", outputs / "comparison.json",
    outputs / "incident.json", explanation, follow_up,
    batch_name="response-batch",
)
print(json.dumps(incident, indent=2))
```

Check the recorded trigger, comparison, actual alias event when selection changed, and fresh batch timestamps. If the candidate failed, say that quality remains unresolved and give the next evidence/containment action; successful CSV delivery does not repair a known error-policy miss. Explain why labels changed the action and why comparison justified selection. The function checks evidence connections; it cannot judge whether your explanation is convincing or your follow-up is useful.

## Watch out for

An alert with no labels supports investigation, not immediate retraining. Small cohorts return `collect_more`. Invalid inputs reject before monitoring. Changed feedback, a stale trigger or a failed candidate must not move the alias. A candidate can remain pending without being useful enough to serve.

If an interactive session closes, reopen the existing workspace and load `candidate.json`, `comparison.json` and the monitor reports from their actual paths. Reconstruct the intended IDs from those retained records; do not fit a replacement merely to recover a Python variable. If a destination already exists, read its retained evidence. Choose a new observation/job name only for a genuinely new execution. Do not recreate evidence merely to obtain another passing picture.

## Quick reference

| Operational need | App function | Evidence to inspect |
|---|---|---|
| Observe two windows | `monitor` | Pinned model, hashes, counts, units and action |
| Fit a justified response | `train_candidate` | Trigger, actual data and pending run/version |
| Compare alternatives fairly | `compare_candidate` | Same validation IDs, errors and improvement |
| Select a qualifying candidate | `promote_candidate` | Rechecked acceptance and actual alias event |
| Deliver predictions | `batch_predict` | Actual output, time and resolved model/source |
| Explain the response | `write_incident` | Connected observations plus your reasoning |

See the [operations API and runbook](https://github.com/richhiey/ml-llm-ops/blob/main/docs/operations.md) for signatures and failure conditions. Your finished work is a monitored, justified model response and a usable new batch, with enough evidence for another engineer to inspect it.

## Learning outcome

Use the provided statistical monitoring scaffold to compare reference and current inference data with one pinned model, apply a declared signal-threshold-action policy, train a candidate only when labeled evidence justifies it, and compare incumbent and candidate on the same separate validation rows.

## Summary

Linden can respond to demonstrated degradation without confusing an alert with permission to train. You have connected explicit incoming data, a declared policy, fair candidate evaluation, a selection decision and actual planner output.


### Inspect the supplied monitoring inputs and policy

Predict the result, execute the actual application call, then inspect its retained evidence. Existing records are reopened; new jobs need distinct output names.

In [ ]:
inputs = ROOT / "docs/course/exercises/monitoring/linden"
operation_dir = project.work / "operations"
operation_dir.mkdir(parents=True, exist_ok=True)
print(json.loads((inputs / "data-card.json").read_text()))
print(json.loads((inputs / "policy.json").read_text()))

### Measure the unlabeled shift with one pinned model

Predict the result, execute the actual application call, then inspect its retained evidence. Existing records are reopened; new jobs need distinct output names.

In [ ]:
path = operation_dir / "monitor.json"
observation = json.loads(path.read_text()) if path.exists() else monitor(workspace, inputs / "reference.csv", inputs / "current.csv", inputs / "policy.json", path)
print(json.dumps(observation, indent=2))
display(plot_monitoring(observation))

### Evaluate the independently labeled observation

Predict the result, execute the actual application call, then inspect its retained evidence. Existing records are reopened; new jobs need distinct output names.

In [ ]:
path = operation_dir / "labeled-monitor.json"
labeled = json.loads(path.read_text()) if path.exists() else monitor(workspace, inputs / "reference.csv", inputs / "current.csv", inputs / "policy.json", path, inputs / "feedback.csv")
print(json.dumps(labeled, indent=2))

### Train the justified candidate and retain its identity

Predict the result, execute the actual application call, then inspect its retained evidence. Existing records are reopened; new jobs need distinct output names.

In [ ]:
assert labeled["action"] == "retrain_candidate", "The observed evidence must justify training"
path = operation_dir / "candidate.json"
if path.exists():
    candidate = json.loads(path.read_text())
else:
    candidate = train_candidate(workspace, operation_dir / "labeled-monitor.json", inputs / "feedback.csv", alpha=1.0)
    write_json(path, candidate)
print(candidate)

### Compare both models on the same validation rows

Predict the result, execute the actual application call, then inspect its retained evidence. Existing records are reopened; new jobs need distinct output names.

In [ ]:
path = operation_dir / "comparison.json"
comparison = json.loads(path.read_text()) if path.exists() else compare_candidate(workspace, operation_dir / "labeled-monitor.json", inputs / "feedback.csv", candidate["version"], path)
print(json.dumps(comparison, indent=2))
roles = pd.read_csv(inputs / "feedback.csv", usecols=["row_id", "partition"])
training_ids = set(roles.loc[roles.partition.eq("train"), "row_id"])
validation_ids = set(roles.loc[roles.partition.eq("validation"), "row_id"])
assert set(comparison["evaluation_rows"]) == validation_ids
assert validation_ids.isdisjoint(training_ids)
print("Same held-fixed validation rows:", len(validation_ids))
print("Error improvement in kWh:", comparison["improvement_kwh"])

### Record the observed reason and select the accepted candidate

Predict the result, execute the actual application call, then inspect its retained evidence. Existing records are reopened; new jobs need distinct output names.

In [ ]:
# The response follows the observed comparison, including a genuine failed-candidate route.
active = workspace.client.get_model_version_by_alias(MODEL_NAME, "active")
if comparison["passed"]:
    reason = (f"Compared the same {len(comparison['evaluation_rows'])} validation rows: "
              f"incumbent {comparison['incumbent_rmse_kwh']:.6f}, candidate {comparison['candidate_rmse_kwh']:.6f} kWh; "
              "both predeclared acceptance conditions pass.")
    intended_version, intended_run = str(candidate["version"]), candidate["run_id"]
    if str(active.version) == intended_version:
        print("Reopened retained candidate selection", active.version)
    else:
        print(promote_candidate(workspace, operation_dir / "comparison.json", reason))
else:
    intended_version, intended_run = str(labeled["model_version"]), labeled["run_id"]
    if str(active.version) != intended_version or active.run_id != intended_run:
        raise ValueError("The incumbent changed; re-evaluate the current state before closing this response.")
    print("Candidate rejected; incumbent retained. Its observed quality miss remains unresolved.")

### Deliver a distinct job and inspect the version it used

Predict the result, execute the actual application call, then inspect its retained evidence. Existing records are reopened; new jobs need distinct output names.

In [ ]:
response_job_name = "response-batch"  # Use a fresh name for a genuinely new execution.
path = workspace.path / f"{response_job_name}.json"
if path.exists():
    job = json.loads(path.read_text())
else:
    _, job = batch_predict(workspace, output_name=response_job_name)
assert job["run_id"] == intended_run and job["resolved_version"] == intended_version, "Keep this older job; choose a fresh job name for the intended version."
assert job["completed_at"] >= comparison["observed_at"], "This job precedes the comparison; retain it and execute a freshly named job."
predictions = pd.read_csv(workspace.path / f"{response_job_name}.csv")
assert predictions.row_id.tolist() == pd.read_csv(DATA / "batch-v1.csv").row_id.tolist()
assert len(predictions) == 24 and np.isfinite(predictions.prediction_kwh).all()
assert job["input_sha256"] == sha256(DATA / "batch-v1.csv") and job["complete_within_60s"]
display(predictions)
print(json.dumps(job, indent=2))

### Record the incident explanation and owned follow-up

Use `write_incident` on your actual monitor/comparison and response batch. Supply your evidence-based explanation and an owner with an observable next check; keep this decision in your own words.

In [ ]:
# Use write_incident provided by this notebook's setup on your actual operation_dir reports.
# State why labels/comparison supported the action, what is still unresolved, and an owner/check.
# Retain the observed incident and inspect its version/timestamps/fresh response_job_name evidence.
pass

## A-S3-05 · Rolling back a bad model version

✂️
# Rolling back a bad model version

Linden Facilities has a working monitoring response. Now decide which incoming observations justify a candidate, reject an unsuitable one and recover an already active model under a changed acceptance requirement. These are different operational decisions.

## What you will work with

Use the supplied [Linden monitoring inputs and policy](https://github.com/richhiey/ml-llm-ops/tree/main/docs/course/exercises/monitoring/linden). They are explicitly synthetic acceptance data. Work in a new investigation store, such as `work/response-investigation`, keeping the cumulative `work/tracking` project unchanged.

Use the already taught `Workspace.run`, registration, observed approval and activation interfaces to establish one guarded incoming baseline in that store. Keep the returned version and source run. Do not run another five-run search. The [operational reference](https://github.com/richhiey/ml-llm-ops/blob/main/docs/operations.md) documents the application interfaces; use explicit store, data, policy and output paths.

For recovery, the exercise intake supplies a separate store at `work/rollback` containing `case.json`, raw runs and registered versions. Open its intake before acting. The [intake instructions](https://github.com/richhiey/ml-llm-ops/blob/main/docs/course/exercises/README.md) distinguish setup of supplied starting records from your recovery task. Reopen an existing intake rather than regenerating it.

## Your task — decide before taking action

Write the expected action and missing/available evidence for each condition:

- Stable reference/current distributions.
- A warmer feature-only window.
- The warmer window with independent `shift-feedback.csv` labels.
- The warmer window with independent `feedback.csv` labels.
- A separate input copy containing only `24` current rows.

Call `monitor` for each condition with the same incoming model and policy, using different new report destinations. Reconcile predictions against actual actions, counts, distances and validation errors. A report with no labels cannot establish demand error. A valid shifted input is not an invalid input.

For the observed labeled retraining trigger, deliberately use `alpha=1000000` as an intentionally unsuitable candidate. Use `train_candidate` and `compare_candidate` with that actual report and feedback file. Inspect both model errors and each declared acceptance criterion before deciding whether the candidate may become active. Retain its failed comparison and prove the alias unchanged. Successful fitting is not a successful response.

Your useful output is a defensible retain/change decision with raw evidence. When the candidate fails, retaining the incumbent prevents that unsupported replacement; it does not erase the measured degradation. Record that unresolved condition and the next evidence or containment action before calling the response complete. Do not edit a `passed` field, change thresholds after seeing scores or open final rows to make a preferred model win.

## Recover an already active version

Inventory `work/rollback` using its actual client/model records. Use `Workspace("work/rollback")` or pass `--state work/rollback` to application commands. Keep every version and artifact.

The revised client requirement is validation RMSE at most `1.6` kWh. Identify the actual active version, each source run, raw validation error and approval status. Explain which previously approved model is suitable now. A prior approval under a looser criterion is insufficient.

Use the real `activate` operation with the suitable actual version and an observed reason, then run a new uniquely named batch against the same isolated store. The [API reference](https://github.com/richhiey/ml-llm-ops/blob/main/docs/api-reference.md) gives argument details. Reconcile the helper's old/new versions, reason and timestamp with the new job's resolved version/source and actual CSV.

Check all `24` IDs once in input order, finite predictions and complete delivery within `60` seconds. Confirm the main project's active version is unchanged. The recovered job is an actual application result; creating the incoming case is not the accomplishment.

## Hints

<details><summary>Hint 1 — Look for missing evidence</summary>

No labels means no observed demand error. Enough independent labels can reveal degradation even when a distribution distance is small. A small monitoring window does not become adequate because it produces a dramatic chart.

</details>
<details><summary>Hint 2 — Separate rejection from recovery</summary>

A pending candidate that never became active can be rejected without rollback. Recovery changes an already active selection to a suitable approved alternative, then verifies a new job.

</details>

## Checkpoints

Your five predictions distinguish the situations using actual evidence. The failed candidate stays pending/inactive. The recovery store ends with a suitable approved version and new valid planner output. Retained history explains the change, and the cumulative project is intact.

<details><summary>Check after recording your decisions and inspecting actual results</summary>

**Result you should get:** stable data continues monitoring; an unlabeled shift and an accurately predicted shifted cohort support investigation; the degraded labeled cohort permits evaluating a candidate; `24` current rows require more observations. The deliberately over-regularized candidate fails acceptance and remains inactive. Its rejection does not restore the incumbent's current-condition quality; that remains an explicit follow-up. In the recovery intake, the alpha-10 trial fails the revised `1.6` gate while the earlier alpha-1 version qualifies.

**How:** inspect action, counts and validation RMSE before interpreting distribution flags. Join each actual version to its raw run and approval. Select the qualifying previous version with a reason, then compare its change event and fresh job records. If you trained without justified labels or confused an old CSV with a new job, preserve that failure and repeat only the missing decision/verification. The helper log is mutable and covers helper-mediated changes; it is not a tamper-proof production audit.

</details>

## Learning outcome

Apply a declared feedback policy to unfamiliar monitoring evidence, retain or qualify a retrained candidate using separate validation data, and restore a suitable previously approved active version in a supplied rollback incident, documenting the resulting audit and new inference evidence.

## Summary

You can distinguish continued investigation, rejection of an inactive candidate and recovery from an already active version. Each decision produces an inspectable system state and useful evidence rather than a copied reference verdict.


In [ ]:
# Predict the five actions before running monitor on separate new reports in an isolated store.
# Declare the intentionally unsuitable candidate, inspect both validation errors and retain its failure.
# Recover work/rollback using actual approved version IDs, then create/inspect a fresh named batch.
# Confirm cumulative workspace selection is unchanged and explain unresolved quality separately.
pass

<details><summary>Review after your own attempt</summary>

**Result you should get:** evidence matching the lesson checkpoints, with an action justified by raw records.

**How:** use the collapsed feedback in the lesson above to inspect your own result. Correct missing or contradicted evidence before advancing; do not substitute a copied reference verdict.

</details>

## A-S3-06 · Registry lifecycle deliverable

✂️
# Registry lifecycle deliverable

Harbor Facilities needs to operate a selected demand model under unfamiliar feedback evidence. It also has an incoming registry recovery incident. The client needs a justified model decision and usable predictions; the newest version is not automatically suitable.

## What you will work with

Continue the same repository in a new isolated investigation store. Use the supplied [Harbor monitoring intake](https://github.com/richhiey/ml-llm-ops/tree/main/docs/course/exercises/monitoring/harbor) and its data card/policy. Its independent synthetic cohorts have separate train/validation/final roles. Establish a guarded incoming baseline using the lifecycle operations you already know, retaining actual IDs and observed validation before approval.

The separate registry intake is supplied at `work/harbor-registry`. Read `case.json` and inspect its three actual registered versions, source runs and tags. [Exercise intake setup](https://github.com/richhiey/ml-llm-ops/blob/main/docs/course/exercises/README.md) is separate from application operation. It creates starting conditions, not a completed response. Keep your cumulative Linden project unchanged and preserve all client artifacts.

## Your task

Before executing a response, inspect the declared policy, cohort sizes and available labels. Record the expected signal, threshold and action. Verify an unlabeled observation first, then evaluate the independent labeled feedback. Retain hashes and the version used for both distributions. Explain why input change, output change and measured degradation support different claims.

When the evidence justifies it, train one candidate. Declare its configuration before comparing it with the incumbent on the same validation IDs. Explain each acceptance criterion using actual errors and improvement. If it fails, retain the incumbent and explicitly record the unresolved measured degradation plus a next evidence/containment action; retaining an approved old model is not proof that it now meets the feedback quality bound. If it passes, use guarded promotion with your own evidence-based reason. In either case, invoke a new client batch and inspect the actual model/source and delivered predictions. Do not change thresholds or inspect final rows to improve your verdict.

## Recover the client's registry selection

The incoming registry client's revised requirement is validation RMSE at most `1.45` kWh. Inspect all three versions and their source validation errors, the current alias and recorded approvals. Choose a suitable previously approved recovery version. An old approval under a looser policy, a high version counter or the pending decoy cannot settle this requirement.

Use the actual application `activate` interface against `--state work/harbor-registry`, supplying the observed version and your reason. Invoke one new, distinctly named batch in that store. Reconcile old/new alias evidence with the new job's source version/run. Preserve the pending version and failed client trial.

## Final check

Your evidence joins monitor → policy → candidate run/data → shared validation comparison → resulting active selection → actual planner output. It also demonstrates a separate rollback with old/new versions, reason, timestamp and a new job using the recovered model. Check the output contract independently: `24` finite, uniquely identified predictions in input order within `60` seconds.

Write a short explanation of one harmful feedback loop to avoid and the evidence that prevents it here. Review your explanation separately from helper success. Reproduction belongs to the later handoff work; this checkpoint does not require another model search or unrelated report.

<details><summary>Check after attempting the client decisions and recovery</summary>

**Result you should get:** Harbor's unlabeled window supports investigation; its independent degraded labels justify evaluating a candidate. Candidate acceptance depends on your actual comparison. A failed candidate leaves the incumbent selected with an unresolved current-condition quality miss; describe that honestly rather than claiming recovery from a passing batch alone. In the registry intake, the previously approved alpha-0.1 version meets `1.45`; the active alpha-10 trial does not, and the alpha-100 decoy is pending. The final recovery output comes from the suitable actual version.

**How:** join each version to its source run, raw validation error and approval before selection. Inspect the real change event and new client batch. Training on model predictions as if they were true demand labels could reinforce its own errors; use independent targets and disjoint validation instead. If you chose the pending decoy or relied on stale inference, preserve that attempt, repair the selection and repeat the missing verification. Copying Linden's guided conclusion cannot establish Harbor's result.

</details>

## Learning outcome

Deliverable: a registered and validated active model, an evidence-backed drift and re-evaluation or retraining decision with its signal, threshold and action, a candidate comparison on separate validation rows, and a demonstrated rollback with audit and fresh inference evidence.

## Summary

Harbor receives a traceable model response and a demonstrated recovery with usable fresh output. Its actual policy, evidence and version identities support the decision while remaining separate from Linden's cumulative project.


In [ ]:
# Use supplied Harbor data/policy in an isolated workspace and record the expected response first.
# Train/compare only when justified; retain or select using the actual shared-validation evidence.
# Recover the supplied Harbor registry under its revised bound and inspect new job/audit identities.
# Produce actual client evidence and explain the feedback loop/current-condition limitations.
pass

<details><summary>Review after your own attempt</summary>

**Result you should get:** evidence matching the lesson checkpoints, with an action justified by raw records.

**How:** use the collapsed feedback in the lesson above to inspect your own result. Correct missing or contradicted evidence before advancing; do not substitute a copied reference verdict.

</details>